# 1. 데이터 등록

## 1.1. 날씨 데이터 조회 (API) 및 등록

### 기상청 동네날씨 예보 조회 (API 연동)

In [41]:
import requests
import urllib3
import os
import pandas as pd
from urllib.parse import unquote
from datetime import datetime
import dotenv

In [42]:
import requests
import urllib3
import os
import pandas as pd
from urllib.parse import unquote
from datetime import datetime
import dotenv

# HTTPS 경고 메시지 무시 설정
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

'''
    1. data.go.kr 사용자 등록 및 API 사용 등록하여 서비스-키를 발급받아야 함
    2. 서비스-키는 .env 파일에 숨겨서 보안성 확보
'''
def get_service_key():
    dotenv.load_dotenv()
    raw_key = os.getenv("SERVICE_KEY")
    return unquote(raw_key)

def get_sky_status(sky_code, pty_code):
    """하늘상태와 강수형태 코드를 조합해 한글 상태 반환"""
    pty_dict = {'1': '비', '2': '비/눈', '3': '눈', '4': '소나기'}
    if pty_code in pty_dict and pty_code != '0':
        return pty_dict[pty_code]
    
    sky_dict = {'1': '맑음', '3': '구름많음', '4': '흐림'}
    return sky_dict.get(sky_code, '-')

def get_weather_forecast(pos=['99', '82']):
    serviceKey = get_service_key() 
    url = "https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst"
    
    # 발표 시각 설정
    now = datetime.now()
    base_times = [2, 5, 8, 11, 14, 17, 20, 23]
    current_hour = now.hour
    past_times = [t for t in base_times if t <= current_hour]
    
    if not past_times:
        base_date = (now - datetime.timedelta(days=1)).strftime('%Y%m%d')
        base_time = "2300"
    else:
        base_date = now.strftime('%Y%m%d')
        base_time = f"{max(past_times):02d}00"

    # 울산 울주군 삼동면 암리 (중산기업) 기준 nx=99, ny=82
    params = {
        'serviceKey': serviceKey,
        'pageNo': '1',
        'numOfRows': '1000',
        'dataType': 'JSON',
        'base_date': base_date,
        'base_time': base_time,
        'nx': pos[0], # '99',
        'ny': pos[1]  # '82'
    }

    try:
        response = requests.get(url, params=params, verify=False)
        res_json = response.json()

        if res_json['response']['header']['resultCode'] == '00':
            items = res_json['response']['body']['items']['item']
            weather_dict = {}

            for item in items:
                time_key = f"{item['fcstDate']} {item['fcstTime'][:2]}:00"
                category = item['category']
                value = item['fcstValue']

                if time_key not in weather_dict:
                    weather_dict[time_key] = {}
                if category in ['TMP', 'REH', 'PCP', 'WSD', 'SKY', 'PTY']:
                    weather_dict[time_key][category] = value

            data_list = []
            for time_str, d in sorted(weather_dict.items()):
                # 강수량(PCP) 숫자 변환 로직
                pcp_raw = d.get('PCP', '강수없음')
                if pcp_raw == '강수없음':
                    pcp_val = 0.0
                else:
                    try:
                        # "1.0mm" 등에서 숫자만 추출
                        pcp_val = float(''.join(filter(lambda x: x.isdigit() or x == '.', pcp_raw)))
                    except:
                        pcp_val = 0.0

                data_list.append({
                    '예보시각': time_str,
                    '상태': get_sky_status(d.get('SKY'), d.get('PTY')),
                    'SKY': d.get('SKY'),
                    'PTY': d.get('PTY'),
                    '기온': float(d.get('TMP', 0)),
                    '습도': int(d.get('REH', 0)),
                    '풍속': float(d.get('WSD', 0)),
                    '강수량': pcp_val
                })
            
            df = pd.DataFrame(data_list)

            # print(f"[{base_date} {base_time} 발표] 울산 삼동면 날씨 예보 (DataFrame)")
            # print("-" * 110)
            # print(df.to_string(index=False)) 
            # print("-" * 110)

            return df
            
        else:
            print(f"API 오류: {res_json['response']['header']['resultMsg']}")
            return None
            
    except Exception as e:
        print(f"데이터 처리 중 오류 발생: {e}")
        return None


In [43]:
if __name__ == "__main__":
    weather_df = get_weather_forecast(['99', '82'])
    # 이제 weather_df를 사용하여 추가적인 데이터 분석이나 엑셀 저장이 가능합니다.

weather_df

,예보시각,상태,SKY,PTY,기온,습도,풍속,강수량
0,20260324 15:00,구름많음,3,0,12.0,50,2.4,0.0
1,20260324 16:00,구름많음,3,0,12.0,50,2.5,0.0
2,20260324 17:00,구름많음,3,0,11.0,50,2.6,0.0
3,20260324 18:00,구름많음,3,0,10.0,60,1.8,0.0
4,20260324 19:00,흐림,4,0,8.0,60,1.3,0.0
...,...,...,...,...,...,...,...,...
61,20260327 12:00,맑음,1,0,20.0,40,1.0,0.0
62,20260327 15:00,맑음,1,0,21.0,20,1.0,0.0
63,20260327 18:00,맑음,1,0,17.0,40,1.0,0.0
64,20260327 21:00,맑음,1,0,13.0,55,1.0,0.0


In [15]:
weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   예보시각    66 non-null     str    
 1   상태      66 non-null     str    
 2   SKY     66 non-null     str    
 3   PTY     66 non-null     str    
 4   기온      66 non-null     float64
 5   습도      66 non-null     int64  
 6   풍속      66 non-null     float64
 7   강수량     66 non-null     float64
dtypes: float64(3), int64(1), str(4)
memory usage: 4.3 KB


### 기상청 데이터 DB 반영

In [44]:
import sqlite3
import pandas as pd

def register_weather(weather_df):
    
    # DB File Path 지정 - DB 변경 시 수정
    db_path = './db/PowerMgt.db'
    df = weather_df.copy()
    
    # [수정] '20260319 15:00' 형태의 문자열에서 직접 추출
    # date: '20260319' 부분 추출 후 '2026-03-19'로 변환
    df['date'] = df['예보시각'].str[:4] + '-' + df['예보시각'].str[4:6] + '-' + df['예보시각'].str[6:8]
    
    # hour: '15:00'에서 앞의 두 글자('15')만 추출하여 정수 변환
    # '20260319 15:00' 구조이므로 index 9부터 2글자가 시간입니다.
    df['hour'] = df['예보시각'].str[9:11].astype(int)
    
    current_now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    df['갱신일시'] = current_now
    
    # DB 테이블 컬럼 순서 (date, hour, temperature, humidity, windspeed, rainfall, status)
    # 입력 DF의 컬럼명이 '기온', '습도' 등 한글인 경우를 가정합니다.
    data_to_db = df[['date', 'hour', '기온', '습도', '풍속', '강수량', '상태', '갱신일시']].values.tolist()
    
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    
    try:
        # PK(date, hour) 중복 시 REPLACE(Update) 실행
        sql = """
        INSERT OR REPLACE INTO WeatherForecast 
        (date, hour, temperature, humidity, windspeed, rainfall, status, update_date) 
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """
        
        cur.executemany(sql, data_to_db)
        conn.commit()
        print(f"성공: {len(data_to_db)}건의 데이터가 등록/업데이트되었습니다.")
        
    except sqlite3.Error as e:
        print(f"데이터베이스 오류: {e}")
        conn.rollback()
        
    finally:
        conn.close()


In [45]:
register_weather(weather_df)

성공: 66건의 데이터가 등록/업데이트되었습니다.


In [6]:
import csv
import sqlite3
from datetime import datetime

def insert_weather_data(csv_file_path):
    # 1. 데이터베이스 연결
    
    # DB File Path 지정 - DB 변경 시 수정
    db_path = './db/PowerMgt.db'
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # 2. 테이블 생성 (이미 생성되어 있다면 생략 가능)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS WeatherForecast (
            date TEXT NOT NULL,
            hour INTEGER NOT NULL,
            temperature REAL NOT NULL,
            humidity REAL NOT NULL,
            windspeed REAL NOT NULL,
            rainfall REAL NOT NULL,
            status TEXT,
            update_date TEXT,
            PRIMARY KEY (date, hour)
        )
    ''')

    # 3. CSV 파일 읽기 및 데이터 삽입
    # 현재 시간을 'YYYY-MM-DD HH:MM:SS' 형식으로 생성
    current_now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    try:
        with open(csv_file_path, 'r', encoding='utf-8') as f:
            # 첫 줄(헤더)은 건너뜁니다
            reader = csv.reader(f)
            next(reader) 

            for row in reader:
                # row 구조: [date, hour, temperature, humidity, windspeed, rainfall, status, update_date]
                # 마지막 update_date(row[7])가 비어있으므로 현재 시간을 넣습니다.
                data = (
                    row[0],        # date
                    int(row[1]),   # hour
                    float(row[2]), # temperature
                    float(row[3]), # humidity
                    float(row[4]), # windspeed
                    float(row[5]), # rainfall
                    row[6],        # status
                    current_now    # update_date (공란 대신 현재 시간 입력)
                )

                # 데이터 삽입 (기본키 중복 시 덮어쓰기: OR REPLACE)
                cursor.execute('''
                    INSERT OR REPLACE INTO WeatherForecast 
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                ''', data)

        conn.commit()
        print("데이터 입력이 완료되었습니다.")

    except Exception as e:
        print(f"오류 발생: {e}")
        conn.rollback()
    
    finally:
        conn.close()

# 실행 예시
# insert_weather_data('weather_data.csv')


In [8]:
insert_weather_data('./data/기상데이터_백업.csv')

데이터 입력이 완료되었습니다.


In [43]:
import pandas as pd
import sqlite3
from datetime import datetime

def register_previous_weather(source_file):
    # 1. 엑셀 파일 읽기 (이미지 내 컬럼명 기준)
    # 실제 파일 내 시트 이름이나 컬럼명에 맞춰 수정이 필요할 수 있습니다.
    df = pd.read_csv(source_file)
    
    db_path = './db/PowerMgt.db'
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    success_count = 0
    
    try:
        for index, row in df.iterrows():
            # '20210101' 형식의 숫자를 '2021-01-01' 문자열로 변환
            raw_date = str(int(row['날짜']))
            formatted_date = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:]}"
            
            # DB insert (REPLACE를 사용하여 중복 시 업데이트)
            sql = """
            INSERT OR REPLACE INTO WeatherForecast (
                date, hour, temperature, humidity, windspeed, rainfall, status, update_date
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """
            
            data = (
                formatted_date,
                int(row['시간']),
                float(row['기온']),
                float(row['습도']),
                float(row['풍속']),
                float(row['강수량']),
                '',              # status (공란)
                current_time      # update_date (현재 시간)
            )
            
            cursor.execute(sql, data)
            success_count += 1
            
        conn.commit()
        print(f"성공: {success_count}건의 데이터가 WeatherForecast 테이블에 저장되었습니다.")
        
    except Exception as e:
        conn.rollback()
        print(f"데이터 입력 중 오류 발생: {e}")
    finally:
        conn.close()

# 실행 예시
# register_previous_weather('날씨데이터.csv')


In [44]:
register_previous_weather('./data/power_estimate_full_2021.csv')


성공: 8760건의 데이터가 WeatherForecast 테이블에 저장되었습니다.


## 1.2. Calendar 연도 별 데이터 입력

In [113]:
import sqlite3
import pandas as pd
import holidays

def register_calendar(year):
    """
    입력받은 연도(year)의 365일(또는 366일) 데이터를 생성하여 
    Calendar 테이블에 저장합니다.
    """
    db_path = './db/PowerMgt.db'
    
    # 1. 해당 연도의 날짜 범위 생성
    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"
    date_range = pd.date_range(start=start_date, end=end_date)
    
    # 2. 한국 공휴일 정보 가져오기
    kr_holidays = holidays.KR(years=year)
    
    calendar_data = []
    
    for dt in date_range:
        # 날짜 정보 추출
        date_str = dt.strftime('%Y-%m-%d')
        y = dt.year
        m = dt.month
        d = dt.day
        
        # 요일 (Pandas: 0:월 ~ 6:일 -> 요청: 1:월 ~ 7:일)
        weekday = dt.weekday() + 1
        
        # 주말 여부 (토:6, 일:7 이면 1, 아니면 0)
        weekend = 1 if weekday >= 6 else 0
        
        # 공휴일 여부 (한국 공휴일 리스트에 있으면 1)
        is_holiday = 1 if dt in kr_holidays else 0
        
        calendar_data.append((date_str, y, m, d, weekday, weekend, is_holiday))
    
    # 3. DB 저장
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    
    try:
        # 중복 방지를 위해 INSERT OR REPLACE 사용
        sql = """
        INSERT OR REPLACE INTO Calendar 
        (date, year, month, day, weekday, weekend, holiday) 
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """
        cur.executemany(sql, calendar_data)
        conn.commit()
        print(f"성공: {year}년 달력 데이터 {len(calendar_data)}건이 등록되었습니다.")
        
    except sqlite3.Error as e:
        print(f"DB 오류: {e}")
        conn.rollback()
    finally:
        conn.close()


In [114]:
register_calendar(2026)

성공: 2026년 달력 데이터 365건이 등록되었습니다.


# 1.3. 한전 요금표 등록

In [116]:
import sqlite3
import pandas as pd

def register_electricity_tariff(file_path):
    
    # DB 파일 경로 지정
    db_path = './db/PowerMgt.db'
    
    # 1. CSV 로드 (첫 번째 컬럼 '시간'을 인덱스로 사용)
    # 이미지 구조상 첫 컬럼은 시간(0~23), 이후 1월~12월 컬럼이 있음
    df = pd.read_csv(file_path)
    
    # 첫 번째 컬럼명을 'hour'로 변경 (만약 '시간' 등으로 되어 있다면)
    df.columns.values[0] = 'hour'
    
    # 2. Matrix 구조를 세로로 풀기 (Unpivot / Melt)
    # 'hour'를 기준으로 각 '월' 컬럼들을 행으로 변환
    df_melted = df.melt(id_vars=['hour'], var_name='month_raw', value_name='bill_rate')
    
    # 3. 데이터 가공
    # '1월', '2월' 문자열에서 숫자만 추출하여 정수형으로 변환
    df_melted['month'] = df_melted['month_raw'].str.replace('월', '').astype(int)
    
    # DB 컬럼 순서에 맞게 정리 [month, hour, bill_rate]
    final_data = df_melted[['month', 'hour', 'bill_rate']].values.tolist()
    
    # 4. DB 저장
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    
    try:
        # 복합키(month, hour) 중복 시 덮어쓰기
        sql = "INSERT OR REPLACE INTO ElectricityTariff(month, hour, bill_rate) VALUES (?, ?, ?)"
        
        cur.executemany(sql, final_data)
        conn.commit()
        print(f"성공: 총 {len(final_data)}건의 요율 데이터가 등록되었습니다.")
        
    except sqlite3.Error as e:
        print(f"DB 오류: {e}")
        conn.rollback()
    finally:
        conn.close()

# --- 실행 ---
# register_electricity_tariff('electricity_tariff.csv')


In [117]:
# --- 실행 ---
register_electricity_tariff('./data/electricity_tariff.csv')

성공: 총 288건의 요율 데이터가 등록되었습니다.


# 1.4. 과거 운영 데이터 입력 

In [45]:
import sqlite3
import pandas as pd

def register_operation_result(csv_path):
    """
    power_estimate.csv 데이터를 읽어 OperationResult 테이블에 저장합니다.
    """
    db_path = './db/PowerMgt.db'
    
    # 1. CSV 로드
    df = pd.read_csv(csv_path)
    
    # 2. 데이터 전처리
    # '예보시각'('2026-03-19 15:00' 형태 문자열 가정) -> date, hour 분리
    df['date'] = df['날짜_시간'].str[:10] # 일자 부분만 가져오기
    
    # 3. 요청하신 컬럼 매칭 (CSV 컬럼명 -> DB 컬럼명)
    # [date, hour, op_code, manpower, output, peak_15, peak_30, peak_45, peak_60, power_usage, bill_rate]
    mapping = {
        '생산구분': 'op_code',
        '공장인원': 'manpower',
        '생산량': 'output',
        '15분': 'peak_15',
        '30분': 'peak_30',
        '45분': 'peak_45',
        '60분': 'peak_60',
        '추정사용전력량': 'power_usage',
        '전기요금(계절)': 'bill_rate'
    }
    
    # 필요한 컬럼만 추출하여 리스트로 변환
    data_to_db = []
    for _, row in df.iterrows():
        data_to_db.append((
            row['date'],
            row['시간'],
            row['생산구분'],
            row['공장인원'],
            row['생산량'],
            row['15분'],
            row['30분'],
            row['45분'],
            row['60분'],
            row['추정사용전력량'],
            row['전기요금(계절)']
        ))

    # 4. DB 접속 및 실행
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    
    try:
        sql = """
        INSERT OR REPLACE INTO OperationResult 
        (date, hour, op_code, manpower, output, peak_15, peak_30, peak_45, peak_60, power_usage, bill_rate) 
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """
        
        cur.executemany(sql, data_to_db)
        conn.commit()
        print(f"성공: {len(data_to_db)}건의 운영 결과 데이터가 처리되었습니다.")
        
    except sqlite3.Error as e:
        print(f"데이터베이스 오류: {e}")
        conn.rollback()
        
    finally:
        conn.close()

# --- 실행 예시 ---
# register_operation_result('power_estimate.csv')


In [46]:
register_operation_result('./data/power_estimate_full_2021.csv')

성공: 8760건의 운영 결과 데이터가 처리되었습니다.


# 2. 데이터 조회 및 활용

# 2.1. 특정 시점의 날씨 조회

In [122]:
import sqlite3
from datetime import datetime

def get_weather_info(target_dt=None):
    """
    지정된 일시(기본값: 현재 시각) 기준 가장 최근 과거의 날씨 정보를 반환
    """
    db_path = './db/PowerMgt.db'
    
    print(target_dt)
    
    # 인자가 없으면 현재 시각 사용
    if target_dt is None:
        target_dt = datetime.now()
    
    target_date = target_dt.strftime('%Y-%m-%d')
    target_hour = target_dt.hour

    try:
        with sqlite3.connect(db_path) as conn:
            conn.row_factory = sqlite3.Row
            cur = conn.cursor()

            # 입력 시점 포함, 가장 가까운 과거 데이터 1건 조회
            query = """
                SELECT * FROM WeatherForecast 
                WHERE date = ? AND hour == ?
            """
            
            cur.execute(query, (target_date, target_hour))
            row = cur.fetchone()
            
            return dict(row) if row else None

    except sqlite3.Error as e:
        print(f"DB Error: {e}")
        return None

In [123]:
# --- 사용 예시 ---
from datetime import datetime

# 문자열을 datetime 객체로 변환
dt_obj = datetime.strptime('2026-03-21 19:00', '%Y-%m-%d %H:%M')

current_weather = get_weather_info(dt_obj)

# 2. 특정 시간 기준 호출 (예: 어제 정오)
# from datetime import timedelta
# yesterday_noon = datetime.now() - timedelta(days=1)
# old_weather = get_weather_info(yesterday_noon)
current_weather

2026-03-21 19:00:00


{'date': '2026-03-21',
 'hour': 19,
 'temperature': 9.0,
 'humidity': 55.0,
 'windspeed': 1.3,
 'rainfall': 0.0,
 'status': '맑음'}

## 2.2 특정일자의 예측을 위한 입력 변수 조회 (1일 24시간 데이터)

In [11]:
import sqlite3
import pandas as pd

def get_prediction_variables_old(target_date):
    """
    날씨 데이터가 없더라도 24시간 행 구조를 유지하며 데이터를 조회합니다.
    """
     
    db_path= './db/PowerMgt.db'
    conn = sqlite3.connect(db_path)
    
    # 입력된 날짜에서 '월' 추출 (예: '2024-05-20' -> '05')
    target_month = int(target_date.split('-')[1])
    
    # print("Target Month :", target_month)
    
    
    
    
    # 0~23시까지의 가상 시간축(Time Backbone)을 생성하여 조인의 기준으로 삼음
    query = f"""
    
    WITH RECURSIVE hours(h) AS (
        SELECT 0 UNION ALL SELECT h + 1 FROM hours WHERE h < 23
    )
    SELECT 
        '{target_date}' AS Date,
        h.h AS hour,
        -- WeatherForecast (데이터 없을 시 NULL)
        W.temperature,
        W.humidity,
        W.windspeed,
        W.rainfall,
        -- OperationForecast
        O.op_code,
        -- O.manpower,
        O.output,
        -- Calendar
        C.weekday,
        C.weekend,
        C.holiday
        -- OperationResult
        -- E.bill_rate  -- 월/시간 기준 요금표에서 조회
    FROM hours h
    LEFT JOIN WeatherForecast W 
        ON W.date = '{target_date}' AND W.hour = h.h
    LEFT JOIN OperationForecast O 
        ON O.date = '{target_date}' AND O.hour = h.h
    LEFT JOIN Calendar C 
        ON C.date = '{target_date}'
    -- LEFT JOIN ElectricityTariff E 
        -- ON E.month = {target_month} AND E.hour = h.h
    ORDER BY h.h ASC;
    """
    
    try:
        df = pd.read_sql_query(query, conn)
        # 만약 공란을 NaN 대신 빈 문자열('')로 출력하고 싶다면 아래 주석 해제
        df = df.fillna('')
        return df
        
    except Exception as e:
        print(f"조회 중 오류 발생: {e}")
        return None
    finally:
        conn.close()

# 사용 예시
# df = get_prediction_variables('2024-05-20')
# print(df)


In [ ]:
import sqlite3
import pandas as pd

# # 일자만으로 예축 (깜깜이 예측)
# features_blind = ['월', '일자', '시간', '요일', '주말여부', '휴일여부'] 
# # 일자 + 날씨로 예축
# features_weather = features_blind + ['기온', '풍속', '습도', '강수량']
# # 일자 + 날씨 + 생산계획을 반영하여 예축
# features_full = features_weather + ['생산구분', '생산량']


def get_prediction_variables(target_date):
    """
    OperationForecast에 해당 날짜 데이터가 없거나 부족하면 
    OperationResult 테이블의 데이터를 대체하여 조회합니다.
    """
    db_path = './db/PowerMgt.db'
    conn = sqlite3.connect(db_path)
    
    # 입력된 날짜에서 '월' 추출
    target_month = int(target_date.split('-')[1])

    query = f"""
    WITH RECURSIVE hours(h) AS (
        SELECT 0 UNION ALL SELECT h + 1 FROM hours WHERE h < 23
    ),
    -- 해당 날짜의 Forecast 레코드 개수 확인
    ForecastCount AS (
        SELECT COUNT(*) as cnt FROM OperationForecast WHERE date = '{target_date}'
    ),
    -- 데이터 소스 결정: Forecast가 12개 미만이면 Result 테이블을 사용
    ActualOp AS (
        SELECT 
            h.h,
            CASE WHEN (SELECT cnt FROM ForecastCount) >= 12 THEN O.op_code ELSE R.op_code END AS op_code,
            CASE WHEN (SELECT cnt FROM ForecastCount) >= 12 THEN O.output  ELSE R.output  END AS output
        FROM hours h
        LEFT JOIN OperationForecast O ON O.date = '{target_date}' AND O.hour = h.h
        LEFT JOIN OperationResult   R ON R.date = '{target_date}' AND R.hour = h.h
    )
    SELECT 
        '{target_date}' AS date, 
        {int(target_date[5:7])} AS month, 
        {int(target_date[8:10])} AS day,
        h.h AS hour,
        C.weekday, C.weekend, C.holiday, 
        W.temperature, W.windspeed, W.humidity, W.rainfall,
        A.op_code,
        A.output
        
    FROM hours h
    LEFT JOIN ActualOp A ON A.h = h.h
    LEFT JOIN WeatherForecast W ON W.date = '{target_date}' AND W.hour = h.h
    LEFT JOIN Calendar C ON C.date = '{target_date}'
    ORDER BY h.h ASC;
    """
    
    try:
        df = pd.read_sql_query(query, conn)
        df = df.fillna('')

        return df
    except Exception as e:
        print(f"조회 중 오류 발생: {e}")
        return None
    finally:
        conn.close()


In [11]:
target_date = '2024-01-25 11:00'      

print(int(target_date[5:7]))
print(int(target_date[8:10]))

1
25


In [18]:
df = get_prediction_variables('2026-03-27')

df

,month,day,hour,weekday,weekend,holiday,temperature,windspeed,humidity,rainfall,op_code,output
0,3,27,0,5,0,0,10.0,1.8,80.0,0.0,,
1,3,27,1,5,0,0,,,,,,
2,3,27,2,5,0,0,,,,,,
3,3,27,3,5,0,0,8.0,1.0,65.0,0.0,,
4,3,27,4,5,0,0,,,,,,
5,3,27,5,5,0,0,,,,,,
6,3,27,6,5,0,0,8.0,1.0,60.0,0.0,,
7,3,27,7,5,0,0,,,,,,
8,3,27,8,5,0,0,,,,,,
9,3,27,9,5,0,0,14.0,1.0,45.0,0.0,,


In [49]:
print("[", df['op_code'].iloc[0], "]")
print(len(df['op_code'].iloc[0]))

[  ]
0


In [127]:
import json
import pathlib

def extract_functions_to_py(ipynb_path, output_py_path=None):
    """
    Jupyter Notebook 파일에서 'def'가 포함된 코드 셀만 추출하여 .py 파일로 저장합니다.
    """
    # 출력 파일명이 지정되지 않으면 원본 파일명 기반으로 생성
    if not output_py_path:
        ipynb_path = pathlib.Path("analysis.ipynb")
        output_py_path = ipynb_path.with_stem(f"{ipynb_path.stem}_functions").with_suffix(".py")

    try:
        with open(ipynb_path, 'r', encoding='utf-8') as f:
            nb_data = json.load(f)

        code_cells = []
        
        # 각 셀을 순회하며 코드 셀이고 'def '가 포함된 경우만 수집
        for cell in nb_data.get('cells', []):
            if cell.get('cell_type') == 'code':
                source = "".join(cell.get('source', []))
                if 'def ' in source:
                    code_cells.append(source)
        
        # 수집된 코드를 파일로 저장
        if code_cells:
            with open(output_py_path, 'w', encoding='utf-8') as f:
                f.write("# Generated from: " + ipynb_path + "\n\n")
                f.write("\n\n".join(code_cells))
            print(f"성공: {len(code_cells)}개의 함수 셀을 '{output_py_path}'에 저장했습니다.")
        else:
            print("알림: 'def'를 포함한 코드 셀을 찾지 못했습니다.")

    except Exception as e:
        print(f"오류 발생: {e}")

# 실행 예시
# extract_functions_to_py('my_analysis.ipynb', 'extracted_utils.py')


In [ ]:
extract_functions_to_py('power_db_management.ipynb', 'power_db_masters.py')

성공: 8개의 함수 셀을 'power_db_operations.py'에 저장했습니다.


In [1]:
import pandas as pd
import sqlite3

def get_daily_result(target_date):
    """
    OperationResult 테이블에서 특정 일자(target_date)의 레코드를 조회하여
    Pandas DataFrame으로 리턴하는 함수입니다.
    """
    
    if not target_date.startswith('2021') : 
        target_date = "2021" + target_date[4:]
        
    # 1. 데이터베이스 연결 (실제 파일 경로로 변경 필요, 예: 'my_database.db')
    db_path= './db/PowerMgt.db'
    conn = sqlite3.connect(db_path)
    
    try:
        # 2. SQL 쿼리 작성 (SQL Injection 방지를 위해 파라미터 바인딩 사용)
        query = "SELECT * FROM OperationResult WHERE date = ?"
        
        # 3. pandas의 read_sql_query를 사용하여 DataFrame으로 직접 읽기
        df = pd.read_sql_query(query, conn, params=(target_date,))
        
    finally:
        # 4. 작업 완료 후 연결 종료
        conn.close()
        
    return df

# 사용 예시
# daily_df = get_daily_result('2020-01-01')
# print(daily_df.head())


In [33]:
daily_df = get_daily_result('2021-03-24')
daily_df

,date,hour,op_code,manpower,output,peak_15,peak_30,peak_45,peak_60,power_usage,bill_rate
0,2021-03-24,0,3,0.483986,136,73,70,69,69,46.23,167.2
1,2021-03-24,1,2,0.121622,54,98,106,119,121,72.36,167.2
2,2021-03-24,2,2,0.332627,157,126,120,113,113,72.84,167.2
3,2021-03-24,3,2,0.329764,154,106,117,121,123,72.83,167.2
4,2021-03-24,4,2,0.077419,36,118,114,121,112,72.27,167.2
5,2021-03-24,5,2,0.559406,226,99,83,106,116,73.17,167.2
6,2021-03-24,6,0,0.000000,0,124,122,112,113,74.80,167.2
7,2021-03-24,7,1,1.623116,969,120,136,169,172,113.24,167.2
8,2021-03-24,8,1,0.719780,524,171,188,185,184,111.06,167.2
9,2021-03-24,9,1,2.075734,1343,154,165,172,156,115.07,167.2


In [20]:

input_df = get_prediction_variables('2021-03-27')
input_df

,month,day,hour,weekday,weekend,holiday,temperature,windspeed,humidity,rainfall,op_code,output
0,3,27,0,6,1,0,11.5,0.2,76.0,0.0,0,0
1,3,27,1,6,1,0,11.2,0.3,79.0,0.0,2,390
2,3,27,2,6,1,0,10.6,0.5,83.0,0.0,2,60
3,3,27,3,6,1,0,9.2,0.0,88.0,0.0,2,4
4,3,27,4,6,1,0,9.0,0.2,91.0,0.0,2,678
5,3,27,5,6,1,0,8.5,0.0,94.0,0.0,2,44
6,3,27,6,6,1,0,8.3,0.7,94.0,0.0,2,31
7,3,27,7,6,1,0,9.0,0.2,94.0,0.0,3,760
8,3,27,8,6,1,0,10.4,0.5,86.0,0.0,0,0
9,3,27,9,6,1,0,11.8,1.2,84.0,0.0,0,0


In [12]:
import numpy as np
import pandas as pd

# 1. 생산 그룹별(생산구분 > 0) 통계량 미리 계산
# 그룹별 Q1(25%), Q3(75%), 최대생산량을 딕셔너리로 저장

stats_1 = {'op_code' : 1, 'mean' : 164.7, '1Q' : 155.0, '3Q' : 174.0, 'min' : 138.0, 'max' : 208.0, 'max_prod' : 9830}
stats_2 = {'op_code' : 2, 'mean' : 111.0, '1Q' : 103.0, '3Q' : 118.0, 'min' :  87.0, 'max' : 138.0, 'max_prod' : 7280}
stats_3 = {'op_code' : 3, 'mean' :  73.5, '1Q' :  64.2, '3Q' :  85.0, 'min' :  50.0, 'max' :  93.0, 'max_prod' : 4406}
stats_4 = {'op_code' : 4, 'mean' :  23.0, '1Q' :  22.0, '3Q' :  24.0, 'min' :  20.0, 'max' :  26.0, 'max_prod' : 4258}

# 리스트로 묶어서 데이터프레임 생성
df_stats = pd.DataFrame([stats_1, stats_2, stats_3, stats_4]).set_index(keys='op_code', drop = True)

IDLE_UPPER_POWER = 50.0
IDEL_MAX_POWER = 150.0
IDEL_MIN_POWER = 15.0

# stats = power_df[power_df['생산구분'] > 0].groupby('생산구분').agg({
#     '최대전력평균': [lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)],
#     '생산량': 'max'
# })
# stats.columns = ['Q1', 'Q3', 'MaxProd']

def calculate_estimated_power(op_code, output, peak_power):
    # [Case 1] 비생산 시 (생산구분 == 0)
    if op_code == 0:
        if peak_power < IDLE_UPPER_POWER :
            val = peak_power * 0.7
        else :
            val = IDLE_UPPER_POWER * 0.7 + (peak_power - IDLE_UPPER_POWER) * 0.6
        return np.clip(val, IDEL_MIN_POWER, IDEL_MAX_POWER) # 최소 15, 최대 150 제한
    
    # [Case 2] 생산 시 (생산구분 == 4 )
    elif op_code == 4 :
        val = peak_power * 0.7
        return max(IDEL_MIN_POWER, val)
    # [Case 3] 생산 시 ( 0 < 생산구분 < 4 )
    else :
        
        group_stat = df_stats.loc[op_code]
        
        # 기본 사용량 : Q1의 70%
        base_power = group_stat['1Q'] * 0.7
        
        # 최대사용한계량 : Q3의 90%
        limit_power = group_stat['3Q'] * 0.9
        
        # 생산량에 따른 증가치 계산
        # 공식: 기본전력량 + (최대사용한계량 - 기본전력량) * (현재생산량 / 최대생산량)
        ratio = output / group_stat['max_prod'] if group_stat['max_prod'] > 0 else 0
        est_val = base_power + (limit_power - base_power) * ratio
        
        # 상한선 적용: 최대전력평균의 90%를 초과할 수 없음
        max_allowable = peak_power * 0.9
        
        est_power = min(est_val, max_allowable)
        return round(est_power, 2)

In [13]:
calculate_estimated_power(1, 2000, 150)

np.float64(118.29)

In [10]:
import sqlite3

def update_power_usage(db_path='./db/PowerMgt.db'):
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # 1. 대상 데이터 조회
        cursor.execute("""
            SELECT date, hour, op_code, output, 
                   (peak_15 + peak_30 + peak_45 + peak_60) / 4.0 as avg_peak 
            FROM OperationResult
        """)
        rows = cursor.fetchall()

        # 2. 데이터별 계산 및 업데이트
        for date, hour, op_code, output, avg_peak in rows:
            # None 값 처리 (데이터가 없을 경우 대비)
            op_code = op_code or 0
            output = output or 0
            avg_peak = avg_peak or 0
            
            # 전력 사용량 계산
            estimated_power = calculate_estimated_power(op_code, output, avg_peak)

            # DB 업데이트
            cursor.execute("""
                UPDATE OperationResult 
                SET power_usage = ? 
                WHERE date = ? AND hour = ?
            """, (estimated_power, date, hour))

        conn.commit()
        print(f"{len(rows)}개의 레코드가 업데이트되었습니다.")

    except sqlite3.Error as e:
        print(f"DB 오류 발생: {e}")
    finally:
        if conn:
            conn.close()


# if __name__ == "__main__":
#     update_power_usage()

In [11]:

if __name__ == "__main__":
    update_power_usage()

8760개의 레코드가 업데이트되었습니다.


## 2.X 특정일자의 Peak 전력 예축하기

In [76]:
features_kor = ['월', '일자', '시간', '요일', '주말여부', '휴일여부', '기온', '풍속', '습도', '강수량', '생산구분', '생산량']
features_eng = ['month', 'day', 'hour', 'weekday', 'weekend', 'holiday', 'temperature', 'windspeed', 'humidity', 'rainfall', 'op_code', 'output']

In [32]:
import joblib

features_kor = ['월', '일자', '시간', '요일', '주말여부', '휴일여부', '기온', '풍속', '습도', '강수량', '생산구분', '생산량']
features_eng = ['month', 'day', 'hour', 'weekday', 'weekend', 'holiday', 'temperature', 'windspeed', 'humidity', 'rainfall', 'op_code', 'output']


def predict_peak(input_df) :

    # 1 입력 데아터를 통한 
    # 1. 저장된 모델 파일 불러오기
    regressor_full_path = "./model/xgb_regressor_full.pkl"
    regressor_weather_path = "./model/xgb_regressor_weather.pkl"
    regressor_blind_path = "./model/xgb_regressor_blind.pkl"

    work_df = input_df.copy()
    # 컬럼명 일괄 변경
    work_df.columns = features_kor

    # 2. 입력데아터에 따라 Regressor 선택
    # 2.1. 생산 데아터 확인
   
    if ( work_df['생산구분'].dtype in ['int64', 'int32'] ) :
        xgb_regressor = joblib.load(regressor_full_path)
        print("Full Model")
    elif ( work_df['기온'].dtype in ['float64', 'float32'] ) :
        xgb_regressor = joblib.load(regressor_weather_path)
        print("Weather Model")
        work_df = work_df.drop(columns=['생산구분', '생산량'], errors='ignore')
    else :
        xgb_regressor = joblib.load(regressor_blind_path)
        print("Blind Model")
        work_df = work_df.drop(columns=['기온', '풍속', '습도', '강수량', '생산구분', '생산량'], errors='ignore')

    # 2. 예측 수행 (X_test 등 예측에 사용할 데이터가 필요합니다)
    # 예: pred = xgb_regressor_weather.predict(X_test_weather)

    print("모델 로드 완료 및 예측 준비가 되었습니다.")
    
    print(work_df.head(20))

    # print('xgb_regressor :', xgb_regressor )

    # print('work_df :', work_df.columns )

    pred_y = xgb_regressor.predict(work_df)

    # '시간' 컬럼 추가 (0부터 23까지)

    pred_df = pd.DataFrame(pred_y, columns=['peak_15', 'peak_30', 'peak_45', 'peak_60'])
    # 시간 컬럼 추가 : 0 ~ 23
    pred_df.insert(0, 'hour', range(24))

    # 정수형으로 반올림하여 회신
    return pred_df.round().astype(int)

In [39]:
input_df = get_prediction_variables('2021-09-12')

input_df

,month,day,hour,weekday,weekend,holiday,temperature,windspeed,humidity,rainfall,op_code,output
0,9,12,0,7,1,0,20.9,1.5,98.0,0.0,0,0
1,9,12,1,7,1,0,20.2,0.6,98.0,0.0,0,0
2,9,12,2,7,1,0,19.8,0.2,98.0,0.0,0,0
3,9,12,3,7,1,0,19.5,1.1,98.0,0.0,0,0
4,9,12,4,7,1,0,19.5,1.0,98.0,0.0,0,0
5,9,12,5,7,1,0,20.2,1.9,98.0,0.0,0,0
6,9,12,6,7,1,0,20.7,2.0,96.0,0.0,0,0
7,9,12,7,7,1,0,22.9,1.3,88.0,0.0,0,0
8,9,12,8,7,1,0,24.3,2.0,79.0,0.0,0,0
9,9,12,9,7,1,0,25.9,1.9,66.0,0.0,0,0


In [40]:
estimate_df = predict_peak(input_df)

estimate_df

Full Model
모델 로드 완료 및 예측 준비가 되었습니다.
    월  일자  시간  요일  주말여부  휴일여부    기온   풍속    습도  강수량  생산구분  생산량
0   9  12   0   7     1     0  20.9  1.5  98.0  0.0     0    0
1   9  12   1   7     1     0  20.2  0.6  98.0  0.0     0    0
2   9  12   2   7     1     0  19.8  0.2  98.0  0.0     0    0
3   9  12   3   7     1     0  19.5  1.1  98.0  0.0     0    0
4   9  12   4   7     1     0  19.5  1.0  98.0  0.0     0    0
5   9  12   5   7     1     0  20.2  1.9  98.0  0.0     0    0
6   9  12   6   7     1     0  20.7  2.0  96.0  0.0     0    0
7   9  12   7   7     1     0  22.9  1.3  88.0  0.0     0    0
8   9  12   8   7     1     0  24.3  2.0  79.0  0.0     0    0
9   9  12   9   7     1     0  25.9  1.9  66.0  0.0     0    0
10  9  12  10   7     1     0  27.3  3.3  60.0  0.0     0    0
11  9  12  11   7     1     0  28.1  2.2  57.0  0.0     0    0
12  9  12  12   7     1     0  27.5  3.5  46.0  0.0     0    0
13  9  12  13   7     1     0  27.3  2.9  51.0  0.0     0    0
14  9  12  14   7  

,hour,peak_15,peak_30,peak_45,peak_60
0,0,27,23,21,21
1,1,22,23,23,24
2,2,27,27,25,27
3,3,27,28,25,28
4,4,27,28,25,28
5,5,28,24,27,26
6,6,29,23,24,24
7,7,21,21,23,22
8,8,26,25,28,22
9,9,26,23,27,22
